# Task Separation

Inspect whether task labels visibly organize one activation representation.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from theoretical.activations.source import activation_path
from theoretical.activations.task_separation.analysis import run

## Setup

In [ ]:
model_id = "LiquidAI/LFM2.5-1.2B-Thinking"
task_name = "snli"
layer = "model.layers.8.feed_forward"
checkpoint_name = None
output_directory = Path("theoretical/activations/task_separation/data")

finetuned_path = activation_path(
    model_id,
    task_name,
    layer,
    task_name=task_name,
    checkpoint_name=checkpoint_name,
)

## Complete projection result

In [ ]:
result = run(task_name, finetuned_path)
result["label_counts"], result["explained_variance_ratio"]

## Label projection

In [ ]:
figure, axis = plt.subplots(figsize=(8, 6))
points = result["points"]
labels = np.asarray(result["labels"])
for label in dict.fromkeys(result["labels"]):
    indices = np.flatnonzero(labels == label)
    axis.scatter(points[indices, 0], points[indices, 1], label=label, alpha=0.7, s=18)
axis.set_title(f"{task_name} activation task separation")
axis.set_xlabel("PC1")
axis.set_ylabel("PC2")
axis.legend()
figure.tight_layout()
plt.show()

## Construct and save completed artifacts

In [ ]:
array_payload = {
    "points": result["points"],
    "explained_variance_ratio": result["explained_variance_ratio"],
    "components": result["components"],
}
metadata_payload = {
    "labels": result["labels"],
    "label_counts": result["label_counts"],
}

output_directory.mkdir(parents=True, exist_ok=True)
np.savez(output_directory / f"{task_name}_task_separation.npz", **array_payload)
with (output_directory / f"{task_name}_task_separation.json").open("w", encoding="utf-8") as file:
    json.dump(metadata_payload, file, indent=2)
    file.write("\n")
figure.savefig(output_directory / f"{task_name}_task_separation.png", dpi=200)